# Historical results: filter-count comparison

Read saved logs without retraining or accessing participant records. Outputs are cleared for publication. Run from the repository root or this notebook's directory. See `docs/METHODOLOGY.md` for evaluation limitations.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'results/raw').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'results/raw').is_dir():
    raise RuntimeError('Open from the repository root or analysis/ directory.')

records = []
for path in sorted((ROOT / 'results/raw').glob('*.log')):
    dataset, filters = path.stem.rsplit('_', 1)
    for line_number, line in enumerate(path.read_text().splitlines(), start=1):
        epoch = re.search(r'^epoch (\d+)', line)
        if not epoch:
            continue
        values = dict(re.findall(r'(loss_tr|err_tr|loss_te|err_te_snt|err_te)=([0-9.eE+\-]+)', line))
        records.append(dict(source=path.name, dataset=dataset, filters=int(filters),
                            line=line_number, epoch=int(epoch.group(1)),
                            **{key: float(value) for key, value in values.items()}))
metrics = pd.DataFrame(records)
evaluations = metrics.dropna(subset=['err_te_snt']).copy()
evaluations['clip_accuracy'] = 1 - evaluations['err_te_snt']
rows = []
for (dataset, filters), group in evaluations.groupby(['dataset', 'filters'], sort=True):
    last = group.iloc[-1]
    best = group.loc[group['clip_accuracy'].idxmax()]
    rows.append(dict(dataset=dataset, filters=filters,
                     test_clips=80 if dataset == 'experiment' else 60,
                     evaluations=len(group), last_epoch=int(last['epoch']),
                     last_clip_accuracy=last['clip_accuracy'],
                     best_epoch=int(best['epoch']),
                     best_clip_accuracy=best['clip_accuracy'], source=last['source']))
summary = pd.DataFrame(rows)
display(summary)


In [ ]:
# Check the committed summary against the raw logs without overwriting it.
saved = pd.read_csv(ROOT / 'results/summary.csv').sort_values(['dataset', 'filters']).reset_index(drop=True)
pd.testing.assert_frame_equal(summary, saved, check_dtype=False, rtol=1e-6, atol=1e-6)
print('Summary matches the raw logs.')


## Learning curves

Rows are kept in recorded order, including repeated epoch-zero entries. Training accuracy is based on augmented minibatches; held-out accuracy is based on full clips. The aggregation levels differ.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, dataset in zip(axes, ['simulation', 'experiment']):
    for filters, group in evaluations[evaluations['dataset'] == dataset].groupby('filters'):
        ax.plot(group['epoch'], group['clip_accuracy'], marker='o', markersize=3,
                label=f'{filters} filters')
    ax.set(title=dataset, xlabel='Epoch', ylabel='Held-out clip accuracy', ylim=(0, 1))
    ax.grid(alpha=0.2)
    ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, dataset in zip(axes, ['simulation', 'experiment']):
    group = evaluations[(evaluations['dataset'] == dataset) & (evaluations['filters'] == 10)]
    ax.plot(group['epoch'], 1 - group['err_tr'], label='Training minibatch accuracy')
    ax.plot(group['epoch'], group['clip_accuracy'], label='Held-out clip accuracy')
    ax.set(title=f'{dataset}: 10 filters', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1))
    ax.grid(alpha=0.2)
    ax.legend()
plt.tight_layout()
plt.show()


## Interpretation

The experiment-derived 10-filter run ends at 76.25% clip accuracy (61/80 clips). More filters did not improve the recorded final result. Synthetic runs have low training error but weak held-out performance. These are small, repeatedly evaluated splits, not an independent comparison of architectures or transfer learning. Human response prediction and auditory-filter estimation were not evaluated.
